# 04 — Quality Checks (exercício)

**Objetivo:** praticar testes de qualidade em SQL.

Execute os notebooks em ordem.

## Onde estamos no pipeline

![Star schema alvo](../docs/images/image.png)

Este é o estado final que vamos atingir ao terminar a sessão: uma fato `fact_report` com chaves estrangeiras para quatro dimensões. Cada notebook contribui com uma camada do caminho até esse esquema.

Posição deste notebook (validação do **silver**):

> Fonte → Bronze → Staging → Dim/Fact → **Checks** → Profiling → Marts/Views → Dashboard

Aqui você não cria tabelas novas — você prova que `fact_report` e as `dim_*` estão saudáveis: PK não nula e única, FK válida para `dim_structured_scope`, ordem temporal coerente entre `created_at` e `disclosed_at`. Em produção, este notebook seria substituído por testes `dbt test` ou Great Expectations.

In [ ]:
# Parametros usados pelo Papermill/Airflow.
run_date = None
project_root = None

from pathlib import Path
import sys

for candidato in [Path.cwd(), *Path.cwd().parents]:
    caminhos_common = [
        candidato / "_common.py",
        candidato / "aula02" / "_common.py",
        candidato / "exercicios" / "_common.py",
        candidato / "sessao-02-data-architecture" / "_common.py",
        candidato / "sessao-02-data-architecture" / "exercicios" / "_common.py",
    ]
    encontrado = next((p for p in caminhos_common if p.exists()), None)
    if encontrado:
        sys.path.insert(0, str(encontrado.parent))
        break
else:
    raise FileNotFoundError("Nao encontrei _common.py a partir do diretorio atual.")

from _common import configurar_paths, conectar_duckdb
paths = configurar_paths(project_root)
globals().update(paths)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Banco DuckDB: {DB_PATH}")


## Exercício
Complete os checks críticos.

In [ ]:
con = conectar_duckdb(DB_PATH)
print("Conexão DuckDB aberta.")

In [ ]:
checks = {}
checks["null_report_id"] = con.sql("""
    SELECT COUNT(*) FROM fact_report WHERE report_id IS NULL
""").fetchone()[0]
checks["duplicate_report_id"] = con.sql("""
    SELECT COUNT(*)
    FROM (
        SELECT report_id
        FROM fact_report
        WHERE report_id IS NOT NULL
        GROUP BY 1
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
checks["bad_time_order"] = con.sql("""
    SELECT COUNT(*) FROM fact_report
    WHERE created_at IS NOT NULL
      AND disclosed_at IS NOT NULL
      AND disclosed_at < created_at
""").fetchone()[0]
checks["missing_asset_keys"] = con.sql("""
    SELECT COUNT(*)
    FROM fact_report f
    LEFT JOIN dim_structured_scope a ON f.asset_id = a.asset_id
    WHERE f.asset_id IS NOT NULL
      AND a.asset_id IS NULL
""").fetchone()[0]
checks
